In [19]:
import pandas as pd

housing = pd.read_csv(r"C:\Users\manue\OneDrive\Ambiente de Trabalho\housing-affordability-portugal\data\raw\portugal_listings.csv", low_memory=False)
income = pd.read_csv(r"C:\Users\manue\OneDrive\Ambiente de Trabalho\housing-affordability-portugal\data\raw\income_by_district.csv")

housing.columns
housing.head()

,Price,District,City,Town,Type,EnergyCertificate,GrossArea,TotalArea,Parking,HasParking,...,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,NumberOfBathrooms
0,780000.0,Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.0,552450.0,0.0,False,...,False,NaN,NaN,NaN,NaN,NaN,120.0,NaN,NaN,0.0
1,223000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,81.0,NaN,NaN,2.0
2,228000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,108.0,NaN,NaN,2.0
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,0.0
4,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,2.0


In [17]:
housing = housing[['District', 'Price', 'Type', 'LivingArea']]

housing.head()

,District,Price,Type,LivingArea
0,Vila Real,780000.0,Farm,120.0
1,Faro,223000.0,Apartment,81.0
2,Faro,228000.0,Apartment,108.0
3,Faro,250000.0,Apartment,114.0
4,Faro,250000.0,Apartment,114.0


In [29]:
housing = housing[['District', 'Price', 'Type', 'LivingArea']]
housing['Price'] = pd.to_numeric(housing['Price'], errors='coerce')
housing['LivingArea'] = pd.to_numeric(housing['LivingArea'], errors='coerce')

housing = housing.dropna(subset=['Price', 'LivingArea'])

housing = housing.reset_index(drop=True)

housing.head()

,District,Price,Type,LivingArea
0,Vila Real,780000.0,Farm,120.0
1,Faro,223000.0,Apartment,81.0
2,Faro,228000.0,Apartment,108.0
3,Faro,250000.0,Apartment,114.0
4,Faro,250000.0,Apartment,114.0


In [37]:
housing['District'] = housing['District'].str.strip()
income['District'] = income['District'].str.strip()

housing['District'] = housing['District'].str.title()
income['District'] = income['District'].str.title()

print("Housing districts:", housing['District'].unique())
print("Income districts:", income['District'].unique())

Housing districts: ['Vila Real' 'Faro' 'Leiria' 'Porto' 'Lisboa' 'Guarda' 'Viseu' 'Coimbra'
 'Castelo Branco' 'Ilha Terceira' 'Setúbal' 'Santarém' 'Évora' 'Braga'
 'Ilha De São Miguel' 'Bragança' 'Beja' 'Aveiro' 'Ilha De Porto Santo'
 'Portalegre' 'Ilha De Santa Maria' 'Viana Do Castelo' 'Ilha Da Madeira'
 'Z - Fora De Portugal' 'Ilha Do Faial' 'Ilha Das Flores']
Income districts: ['Lisboa' 'Setubal' 'Porto' 'Aveiro' 'Braga' 'Coimbra' 'Faro' 'Leiria'
 'Viseu' 'Evora' 'Beja' 'Santarem' 'Castelo Branco' 'Portalegre' 'Guarda'
 'Viana Do Castelo' 'Braganca' 'Vila Real' 'Madeira' 'Azores']


In [39]:
district_map = {
    'Évora': 'Evora',
    'Bragança': 'Braganca',
    'Setúbal': 'Setubal',
    'Santarém': 'Santarem',
    'Ilha Terceira': 'Azores',
    'Ilha De São Miguel': 'Azores',
    'Ilha De Porto Santo': 'Madeira',
    'Ilha De Santa Maria': 'Azores',
    'Ilha Do Faial': 'Azores',
    'Ilha Das Flores': 'Azores',
}

housing['District'] = housing['District'].replace(district_map)
housing = housing[housing['District'].notnull()]

In [41]:
merged = pd.merge(housing, income, on='District', how='left')

missing_count = merged['AvgMonthlyIncome'].isnull().sum()
print(f"Number of listings with missing income: {missing_count}")

Number of listings with missing income: 50


In [45]:
median_income = merged['AvgMonthlyIncome'].median()
merged['AvgMonthlyIncome'] = merged['AvgMonthlyIncome'].fillna(median_income)

missing_count = merged['AvgMonthlyIncome'].isnull().sum()
print(f"Number of listings with missing income: {missing_count}")

Number of listings with missing income: 0


In [63]:
merged['AnnualIncome'] =  merged['AvgMonthlyIncome'] * 12
merged['PriceToIncome'] = merged['Price'] / merged['AnnualIncome']

merged['PricePerM2'] = merged['Price'] / merged['LivingArea']

merged.head(10)

,District,Price,Type,LivingArea,AvgMonthlyIncome,AnnualIncome,PriceToIncome,PricePerM2
0,Vila Real,780000.0,Farm,120.0,1295.0,15540.0,50.193050,6500.000000
1,Faro,223000.0,Apartment,81.0,1314.0,15768.0,14.142567,2753.086420
2,Faro,228000.0,Apartment,108.0,1314.0,15768.0,14.459665,2111.111111
3,Faro,250000.0,Apartment,114.0,1314.0,15768.0,15.854896,2192.982456
4,Faro,250000.0,Apartment,114.0,1314.0,15768.0,15.854896,2192.982456
5,Faro,250000.0,Apartment,115.0,1314.0,15768.0,15.854896,2173.913043
6,Faro,2950000.0,Building,406.0,1314.0,15768.0,187.087773,7266.009852
7,Faro,9500.0,Apartment,27.0,1314.0,15768.0,0.602486,351.851852
8,Faro,158000.0,Apartment,42.0,1314.0,15768.0,10.020294,3761.904762
9,Faro,250000.0,Apartment,85.0,1314.0,15768.0,15.854896,2941.176471
